In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
@dlt.table(
    name = "stage_bookings"
)
def stage_bookings():
    df = spark.read.format("delta")\
          .load("/Volumes/workspace/bronze/bronzevolume/bookings/data/")
    
    return df
# this will load the data incrementally and this is the staging area and we don't perform any transformations here

In [0]:
@dlt.view(
    name = "trans_bookings"
)
def trans_bookings():
    df = spark.readStream.table("stage_bookings")
    df = df.withColumn("amount", col("amount").cast(DoubleType()))\
           .withColumn("modifiedDate", current_timestamp())\
           .withColumn("booking_date", to_date(col("booking_date")))\
           .drop(col("_rescued_data"))
    return df
# this contain transformations that we did in the data.

In [0]:
rules = {
            "rule1" : "booking_id IS NOT NULL",
            "rule2" : "passenger_id IS NOT NULL"
        }

In [0]:
@dlt.table(
    name = "silver_bookings"
)
@dlt.expect_all_or_drop(rules)
def silver_bookings():
    df = spark.readStream.table("trans_bookings")
    return df
# expect_all_or_drop: it will drop all the rows that don't follow the rules.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/bronze/bronzevolume/flights/data/")

df = df.withColumn("flight_date", to_date(col("flight_date")))\
       .drop(col("_rescued_data"))\
       .withColumn("modifiedDate", current_timestamp())
display(df)

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/bronze/bronzevolume/customers/data/")

df = df.drop(col("_rescued_data"))\
       .withColumn("modifiedDate", current_timestamp())
display(df)

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/bronze/bronzevolume/airports/data/")

df = df.drop(col("_rescued_data"))\
       .withColumn("modifiedDate", current_timestamp())
display(df)